# 掼蛋·扑克牌识别模型训练（Colab 一键版）

本笔记本在 **Google Colab 免费 GPU** 上训练识别扑克牌「数字+花色」的模型（YOLO）。
练完得到模型文件，下载回来接到掼蛋 App：摄像头识别 → 自动组牌 → 出牌建议。

**用法**：菜单「运行时 → 更改运行时类型 → 选 GPU」，再「运行时 → 全部运行」。


## 1. 确认已分到 GPU


In [ ]:
!nvidia-smi

## 2. 安装训练框架（Ultralytics YOLO）


In [ ]:
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3. 获取扑克牌数据集 ⚠️必须改这格

**不要直接运行下面的示例**——里面的 workspace/project 是占位符，会报 `BadZipFile`。
正确做法：
1. 打开 https://universe.roboflow.com ，搜索 `playing cards`，选一个数据集；
2. 点 **Download Dataset → 格式选 YOLOv8 → Show download code**；
3. 把它给你的**整段代码**复制，替换掉下面这格的全部内容，再运行。
（那段代码里已含正确的 workspace/project/version 和你的 api_key。）


In [ ]:
# ↓↓↓ 用 Roboflow 数据集页面的『Show download code』整段替换这里 ↓↓↓
from roboflow import Roboflow
rf = Roboflow(api_key="把你的API_KEY粘到这里")
project = rf.workspace("替换为真实workspace").project("替换为真实project")
dataset = project.version(1).download("yolov8")   # 版本号按页面给的改
# ↑↑↑ 用真实代码替换以上几行 ↑↑↑
print('数据集下载到:', dataset.location)

## 4. 定位数据集配置并开始训练

自动查找 `data.yaml`（下载成功才会有），避免路径写错。`epochs` 先 50 试跑。


In [ ]:
import glob
yamls = glob.glob('**/data.yaml', recursive=True)
assert yamls, '没找到 data.yaml —— 说明第3格数据集没下成功，请用数据集页面的下载代码替换第3格后重跑。'
DATA = yamls[0]
print('使用数据集配置:', DATA)

from ultralytics import YOLO
model = YOLO('yolov8n.pt')   # 要更准可换 yolov8s.pt / yolov8m.pt
results = model.train(data=DATA, epochs=50, imgsz=640, batch=16,
                      patience=20, project='guandan_cards', name='exp')

## 5. 看效果（验证集指标）


In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)

## 6. 导出模型（给 App 用）


In [ ]:
best = 'guandan_cards/exp/weights/best.pt'
m = YOLO(best)
m.export(format='onnx')
print('模型：', best, ' 和 同目录 best.onnx')

## 7. 下载模型到本地


In [ ]:
from google.colab import files
files.download('guandan_cards/exp/weights/best.pt')
files.download('guandan_cards/exp/weights/best.onnx')

## 8. 拿回模型后怎么用

把 `best.pt` 放到本仓库 `vision/`，运行 `python vision/recognize.py 照片.jpg`，
会输出识别到的牌并直接调用掼蛋引擎给『自动组牌 + 出牌建议』。
手机/眼镜端用 `best.onnx`（可转 TF.js 网页跑 / TFLite 原生跑）。
